# 07. 예측한 행동을 적용하고 다시 관측하기

**목적:** 실제 가상 로봇에 행동을 적용하는 폐루프를 실행하고, 재관측 전 실행 길이 10과 5의 차이를 비교합니다.

예상 시간 90분. [전체 안내](README.md).

![07. 예측한 행동을 적용하고 다시 관측하기의 전체 과정](../교재그림/vla-rollout-process.svg)

06에서는 행동을 계산만 했습니다. 07에서는 **예측 → 행동 적용 → 바뀐 관측 → 다시 예측** 을 반복합니다. 로봇이 물체를 집고 놓는 동작과 환경의 성공 판정을 처음 함께 확인하는 단원입니다.

### 06과 달라지는 부분

07은 `infer-01`에 저장된 50개를 재생하지 않습니다. 자체 환경을 초기화하고 모델을 로드한 뒤 현재 관측으로 새 행동을 예측합니다. 50개 중 앞 `execute`개만 하나씩 적용하며, 계속 진행하면 새 관측으로 다음 50개를 예측합니다. 초기 배치로 되돌리는 reset은 묶음마다 반복하지 않습니다.

### 두 반복문의 역할

| 실행 부분 | 역할 |
|---|---|
| 바깥 `while` | 현재 관측으로 입력을 준비하고 행동 묶음을 새로 예측 |
| 안쪽 `for` | 묶음 앞부분의 행동을 하나씩 적용 |
| `env.step(...)` | 명령을 적용해 물리 상태를 갱신하고 관측·종료 정보를 받음 |
| 매 step의 성공·종료 검사 | 성공 또는 종료 조건이면 남은 행동도 실행하지 않고 멈춤 |

기본은 미래 50개 중 앞 10개입니다. 실행 후 남은 40개를 이어 쓰는 대신 최신 관측에서 새로 예측합니다. `execute=5`이면 앞 5개 후 다시 예측합니다. **카메라의 물리적 촬영 fps를 바꾸는 설정이 아니라 정책에 새 관측을 넣는 간격을 바꾸는 설정** 입니다.

### 기준 실행에서 무엇을 보나요?

설정 셀 다음 예제 셀 전체를 실행합니다. `seed7-h10` 결과가 생성된 뒤 같은 폴더의 영상을 표시합니다. 같은 조건의 완료 결과가 있으면 재사용합니다. 영상에는 실제로 환경에 적용한 행동의 결과가 보이지만, 저장 파일의 20fps 재생 속도는 실제 추론 속도가 아닙니다.

| 출력 | 읽는 법 |
|---|---|
| `status` | 실행 절차가 완료인지, 예외·중단으로 미완료인지 |
| `success` | 환경이 과제 성공을 판정했는지 |
| `steps` | 실제 적용한 행동 수 |
| `termination_reason` | success·step_limit·terminated·truncated·오류 등 종료 이유 |
| `len(chunk_seconds)` | 실제 정책 묶음 추론 횟수 |
| `execution_seconds` | 환경 준비·모델 로드를 제외한 실행 구간 시간 |

`completed`이면서 `success=false`일 수 있습니다. 예를 들어 최대 스텝에 도달해 정상 종료했지만 과제를 달성하지 못한 경우입니다. 예외·사용자 중단은 별도 미완료로 분리합니다. 성공 판정과 함께 영상에서 대상 선택, 접근, 잡기, 옮기기, 놓기의 어느 과정이 달랐는지 살펴봅니다.

### 실행 계획 예제는 실제 로봇 실험이 아닙니다

<!-- procedure:lesson-43-e862f8:start -->
![행동 실행에서 비교 영상까지](../교재그림/절차/lesson-43.svg)

종료 전이면 2번으로 반복합니다. 6번의 비교는 초기 배치·seed·모델을 유지하고 실행 길이만 바꿉니다.
<!-- procedure:lesson-43-e862f8:end -->


`execution_plan(23,10)`의 `[10,10,3]`은 23개를 최대 10개씩 나누는 **계산 예제** 입니다. 환경을 실행하지 않으며 성공에 따른 조기 종료도 반영하지 않습니다. 실제 실행의 마지막 묶음은 성공·종료 또는 남은 스텝 제한으로 더 짧아질 수 있습니다.

### TODO: 비교 결과 생성부터 기록까지

1. 기준 `seed7-h10` 결과와 영상을 먼저 확인하고 보존합니다.
2. 기존 `notes` 셀에서 아래 호출을 실행합니다. 영상 경로만 바꾸지 말고 생성 호출을 먼저 완료합니다.

```python
comparison_name = 'seed7-h5'
comparison = run_lesson('rollout', comparison_name,
                        seed=7, execute=5, steps=280, initial_state=0)
display(Video(url='/files/' + (RUNS/comparison_name/'rollout.mp4')
              .relative_to(ROOT).as_posix(), embed=False))
```

3. 기준 `result`와 `comparison`의 추론 횟수·실행 시간·스텝 수·완료·성공·종료 이유를 기록합니다. task·seed·배치·모델은 유지하고 execute만 바뀌었는지 확인합니다.
4. 두 영상에서 차이가 드러나는 장면과 파일명을 기록합니다. 시간 표시는 저장 영상 안의 위치이며 실제 처리 시간과 구분합니다. 차이가 없으면 없다고 기록합니다.

**상세 해설:** 같은 길이를 실행한다면 h5가 h10보다 더 자주 추론할 수 있지만 실제 호출 횟수는 성공·종료 시점에도 영향을 받습니다. h5가 항상 두 배 호출하거나 반드시 더 성공하는 것은 아닙니다. 정책의 예측, 장면 변화, 종료 스텝을 함께 보고 해석해야 합니다. 이번 동기식 실행에서는 추론 중 환경이 기다립니다.

새 재시도는 미사용 이름을 쓰고 08의 실험 목록에도 그 이름과 조건을 대응시킵니다. 기존 중단·실패를 삭제해서 비교 결과를 좋게 만들지 않습니다. 별도 최대 10스텝 실험은 핵심 h10/h5 비교와 분리하며 종료 이유를 실제 결과에서 확인합니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'session-01'`은 같은 회차이며 결과는 `Data/logs/day5/session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.

## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

<!-- procedure:novnc-1744bd:start -->
![noVNC의 정상 대기와 완료](../교재그림/절차/novnc.svg)

03의 준비 검사와 05의 저장 배열 읽기는 새 실험 창을 열지 않습니다. 기존 완료 결과 재사용에도 새 창이 없습니다.
<!-- procedure:novnc-1744bd:end -->


JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

**이 노트북에서는 로봇이 실제로 행동하는 장면을 봅니다.** 추론 중에는 환경이 대기하고, ACTION EXECUTION에서 예측한 행동을 적용합니다. 스텝 수와 추론 횟수의 변화를 함께 관찰하세요.

설정 셀의 `GUI`는 자동으로 켜집니다. 창 없이 시간 비교를 할 때는 `GUI = False`로 바꾸고 설정 셀을 다시 실행합니다. CLI 대체 실행은 `run_episode.py`에 `--gui`를 붙입니다. 노트북과 CLI를 동시에 실행하지 않습니다.

환경 초기화가 끝난 뒤 실제 초기 장면에서 창이 열립니다. 초기화 중 오류는 노트북 로그에서 확인합니다. 화면에는 실행 이름·모드·현재 상태·스텝·추론 횟수·적용 길이·직전 추론 초가 표시됩니다.

| 화면 상태 | 의미 |
|---|---|
| `INITIAL OBSERVATION` | 초기 관측 준비 완료 |
| `MODEL LOADING` | 모델 로드 중, 환경은 대기 |
| `INFERENCE` | 행동 계산 중, 환경은 대기 |
| `ACTION EXECUTION` | 예측 명령을 환경에 적용 중 |
| `FINISHED` | 종료 이유와 성공 여부 표시. 마지막 장면이며 더 움직이지 않음 |
| `INCOMPLETE / ERROR` | 미완료. 노트북의 원인 로그 확인 |

**정상 종료 화면을 관찰한 뒤 창을 닫으면 셀이 끝납니다.** 결과와 영상은 종료 화면 대기 전에 저장됩니다. 실행 도중 창을 닫으면 다음 안전한 처리 지점에서 사용자 중단으로 기록합니다. GPU 추론·모델 로딩 중에는 그 작업이 반환될 때까지 중단 반영이 늦을 수 있습니다. 창 조작은 표시용 복사본에만 적용되어 정책 카메라와 실제 실험 상태를 바꾸지 않습니다.

**기존 완료 결과를 읽으면 새 GUI는 열리지 않습니다.** 새 시연은 사용하지 않은 실행 이름으로 수행하고 변수·파일 경로도 함께 맞춥니다. GUI 사용 여부는 `result.json`의 `gui_enabled`에 남습니다. 화면 표시 비용이 있으므로 GUI 실행과 비GUI 실행의 시간을 같은 조건처럼 비교하지 않습니다.


**브라우저 탭을 닫아도 실험은 종료되지 않습니다.** 다시 같은 주소로 접속하면 진행 중 화면으로 돌아옵니다. FINISHED 후 화면 안의 MuJoCo 창을 닫아야 셀이 끝납니다. noVNC 연결과 탭은 다음 실험까지 유지하세요. 다른 학생 화면이 보이면 포트를 확인합니다.


## 슬라이드와 함께 진행하기

강사의 목적·흐름 설명을 들은 뒤 해당 셀을 실행하고, 출력과 TODO를 슬라이드의 확인 질문으로 해석합니다. 노트북 사전 정독을 전제하지 않습니다.

쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **기준 실행·horizon 5 TODO·검증** → [5일차 슬라이드 43쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s039) · [배포 PDF 43쪽](../슬라이드/5일차.pdf#page=43)에서 49쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이전 자료의 위치와 필요한 폐루프 계약

[06_inference.ipynb의 「예제」](06_inference.ipynb)는 한 관측에서 `(1,50,7)` 행동을 저장합니다. 여기서 필요한 내용은 각 묶음이 미래 50시점의 7개 성분을 예측하지만 환경에는 앞 execute개만 적용한 뒤 새 관측으로 나머지를 대체한다는 점입니다. [05_observation_action.ipynb의 「개념」](05_observation_action.ipynb)에 상태8·영상2 전처리가 있으며 실행 스크립트가 같은 변환을 수행합니다.

이 노트북은 이전 infer-01 행동 파일을 이어 실행하지 않고 자체 reset·모델 로드·관측으로 새 에피소드를 수행합니다. 따라서 04–06 산출물이 없어도 필수 배포 모델·캐시만 있으면 실행 가능합니다. 기본 task0/initial_state0/seed7/steps280을 고정하고 `seed7-h10`과 TODO의 `seed7-h5`를 같은 SESSION에 저장합니다. 각 폴더의 result.json, rollout.mp4, action-chunks.npy, 초기 영상·관측이 다음 평가의 근거입니다.

[run_episode.py](scripts/run_episode.py)의 while은 재관측 반복, 안쪽 for는 묶음 앞부분 실행, `env.step`의 `is_success`는 과제 성공입니다. `execution_seconds`는 모델 로드·환경 준비를 제외한 실행 시간이고 `len(chunk_seconds)`는 실제 정책 호출 횟수입니다. 저장 영상20fps는 벽시계 처리 속도가 아닙니다.

## 흐름 그림

![예측한 나머지 40개를 이어 쓰지 않고, 앞 10개 실행 후 새 관측으로 다시 예측합니다.](../교재그림/vla-horizon.svg)

예측한 나머지 40개를 이어 쓰지 않고, 앞 10개 실행 후 새 관측으로 다시 예측합니다.


## 개념

기준 실행은 seed7, task0, 최대280스텝, 실행 horizon10입니다. 50개 행동 중 앞10개만 사용한 뒤 새 관측으로 다시 예측합니다. 각 행동은 환경 범위 [-1,1]로 제한합니다. `is_success`, terminated, truncated, 최대스텝 중 종료 조건이 생기면 멈춥니다.

status=completed와 success=true를 구분하세요. 예외·중단은 incomplete이며 success=false로 기록합니다. 마지막 영상 프레임과 환경 성공 판정을 함께 확인하세요.

### 단계별 활동

진행: 제어 루프 읽기15분 → 기준 실행20분 → horizon 변경 TODO20분 → 영상·로그 분석20분 → 중단·복구·결과 기록15분. run_episode.py에서 while/for, env.step, is_success를 찾으세요. 모델이 50개를 예측했는데 10개만 실행하는 위치를 확인합니다. 최대스텝을 10으로 낮춘 별도 실행으로 정상적인 스텝한도 종료와 오류 종료의 차이를 체험하세요.

In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA').is_dir())
SCRIPTS = ROOT / '10.VLA/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state, gui=GUI)


## 예제

### 실행 전 확인 — FINISHED 후 MuJoCo 창 닫기

**noVNC에서 `FINISHED`를 확인하면 화면 안의 MuJoCo 창 오른쪽 위 ×를 눌러 닫아주세요.** 창을 닫아야 아래 셀이 완료되고 다음 단계로 진행됩니다. 결과는 이미 저장되었지만, 마지막 장면을 관찰하도록 창을 유지하므로 노트북의 `[*]`와 경과 시간은 계속 표시됩니다. **브라우저 탭만 닫으면 셀은 끝나지 않습니다.** 실행 중 창을 닫으면 중단으로 처리되므로 `FINISHED`를 확인한 뒤 닫습니다.

행동 실행이 끝난 뒤 완료 창을 닫으면 노트북에 저장 동영상과 실행 결과가 표시됩니다. 이후 비교 실험에서도 같은 순서로 진행합니다.

기존 완료 결과를 재사용하거나 `GUI=False`로 실행하면 새 GUI 창이 열리지 않으며, 창 닫기를 기다리지 않습니다.


In [ ]:
result = run_lesson('rollout', 'seed7-h10', execute=10)
from IPython.display import Video, display
display(Video(url='/files/' + (RUNS/'seed7-h10/rollout.mp4').relative_to(ROOT).as_posix(), embed=False))
print(result['status'], result['success'], result['steps'], result['termination_reason'])

### 입력과 실행 흐름 확인

In [ ]:
def execution_plan(max_steps, execute):
    return [min(execute, max_steps-start) for start in range(0,max_steps,execute)]
print('23스텝, horizon10:', execution_plan(23,10))
assert execution_plan(23,10) == [10,10,3]

## TODO

execute를 5로 바꾸어 새 이름 seed7-h5로 실행하세요. 추론 횟수, 총 실행 시간, 성공 여부를 비교하고 실패 영상에서 언제 차이가 발생했는지 기록하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**작성 위치와 출력:** 바로 아래 코드 셀의 `notes = {}`에 관찰 답안을 dict로 작성하고 노트북을 저장합니다. 새 정책 함수를 구현하는 TODO가 아닙니다. 코드 검토만 할 때는 아래 형식의 의미만 읽고, 실제로 관찰하지 않은 값은 `None`/`미측정`으로 남깁니다. notes 출력은 파일 저장이 아니므로 답안은 셀에 직접 적고 노트북 자체를 저장하세요.
**입력 자료:** seed7-h10과 seed7-h5의 result.json·rollout.mp4.
**권장 답안 형식** (key 이름은 기록용 권장안이며 자동 채점 schema가 아닙니다):
```python
notes = {'baseline_run': 'seed7-h10',
 'comparison_run': 'seed7-h5',
 'inference_count_h10': None,
 'inference_count_h5': None,
 'execution_seconds_h10': None,
 'execution_seconds_h5': None,
 'success_h10': None,
 'success_h5': None,
 'difference_at': '미측정',
 'evidence_status': '미측정'}
```
**진행 힌트:** 실습 때 `run_lesson("rollout","seed7-h5",execute=5)`로 비교하고 seed/초기배치/최대 step은 유지합니다. 두 result.json에서 추론 횟수=len(chunk_seconds), execution_seconds, success, steps를 비교하세요. execution_seconds는 모델 로드·초기화 제외 시간이며 총 수업 대기시간과 다릅니다. 영상 경로도 바꾸고 차이가 드러나는 시각과 행동을 기록합니다. 읽기 검토에서는 호출하지 않습니다.
**강의 멘트:** “horizon5가 더 자주 관측해도 더 성공한다고 미리 결론내리지 않습니다. 완료·성공·지연을 별도로 비교합니다.”
**완료 확인:** 각 항목에 근거와 이유가 있고 실제 관찰/실측 값과 개념 설명을 구분했는지 확인합니다. 정답 문단의 수치는 관찰하지 않은 자기 실행 결과를 대신하지 않습니다.
<!-- todo-review:requirements:end -->

**작성 순서와 확인:** 바로 아래 답안 셀에서 `comparison = run_lesson('rollout','seed7-h5',seed=7,execute=5,steps=280,initial_state=0)`를 실행하고 원래 `result`(h10)와 comparison(h5)를 비교합니다. 파일은 `RUNS/'seed7-h5/result.json'`, 영상은 `RUNS/'seed7-h5/rollout.mp4'`입니다. 예제의 Video 표시 경로만 이 이름으로 바꾸면 같은 노트북에서 볼 수 있습니다. 실행 횟수=len(chunk_seconds), 실행초=execution_seconds, 준비초=environment_setup_seconds, 로드초=model_load_seconds를 구분하세요. 실패가 없으면 차이 시점을 억지로 만들지 말고 “두 영상에서 확인한 차이 없음”과 관찰 범위를 적습니다.

별도 스텝한도 활동은 `run_lesson('rollout','limit10-h10',execute=10,steps=10)`을 사용합니다. 기존 seed7-h10 조건을 덮어쓰지 않고 종료 이유가 step_limit인지 success/terminated/truncated인지 실제 결과로 확인합니다. 모든 조건을 기록하고 h10/h5의 영상·수치·완료/성공을 대응시키면 핵심 TODO 완료입니다.


In [ ]:
# 관찰 결과/실험 설정을 여기에 기록하세요.
notes = {}
notes

## 검증과 정답·복구

입력/모델/seed를 유지하고 horizon만 바꿉니다. 작은 horizon은 재관측을 늘리지만 wall time이 증가할 수 있습니다. 성공 증가를 미리 가정하지 않습니다. 메모리 오류는 다른 모델 프로세스 종료 후 재실행합니다.

TODO 완료 예시(완료 결과가 있으면 같은 조건인지 확인하고 읽습니다):
```python
comparison = run_lesson('rollout', 'seed7-h5', execute=5)
```

파일·캐시·CUDA 오류는 [복구 안내](README.md#복구)와 이 노트북 시작 절에 있습니다. 필요한 복구는 전용 이미지·/opt/course 커널 확인, Data 마운트·배포 모델/캐시 존재 확인, 다른 GPU 작업 종료 후 한 프로세스 실행입니다. 누락 파일을 다운로드하거나 상태8을 잘라 우회하지 않습니다. CLI는 노트북과 같은 실험의 대체 실행입니다. 같은 조건의 완료 결과는 다시 읽습니다. 새 실험은 SESSION과 후속 노트북의 회차를 함께 바꾸세요. 실험 출력은 삭제하지 않습니다.

### 정답 해설 — 입력부터 결과까지

**한 조건만 변경:** 기준 seed7-h10과 비교 실행 seed7-h5는 과제·초기 배치·seed·모델을 같게 두고 execute만 10→5로 바꿉니다. 출력 이름도 바꿔 기준 결과를 보존합니다.

```text
관측 A → 미래 50개 예측 [1..10 | 11................50]
                         실행       미실행
                            ↓ 새 관측 B
         새 50개 예측 [1..10 | 11................50]
                         실행       미실행
```

기존 나머지 40개를 계속 이어 쓰지 않고 새 관측으로 바꿉니다. execute=5이면 더 자주 다시 보지만 정책 호출 비용도 늘어날 수 있습니다. 남은 23스텝을 최대 10개씩 쓰면 [10,10,3]이며 마지막 3개가 모자란 묶음이라는 오류는 아닙니다.

**확인 질문:** 더 자주 재관측해도 성공이 반드시 늘지 않는 이유를 자신의 영상·시간 결과에서 찾아보세요.

**적용·재확인:** 관찰 답안은 자신의 실제 파일·조건으로 작성합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 기준10과 변경5의 로그·영상·완료/성공 구분; 같은 SESSION 유지

[5일차 슬라이드 43쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s039) · [배포 PDF 43쪽](../슬라이드/5일차.pdf#page=43)로 돌아가 **43–49쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

